In [ ]:
import time
import threading
import ctypes as ct
import sys
sys.path.append('D:\\nbmlab\\application\\robotic\\scanCONTROL Windows SDK 4.1.1\\C++ SDK (+python bindings)\\python_bindings\\pyllt')
import numpy as np
from matplotlib import pyplot as plt
import matplotlib.animation as animation
import pyllt as llt


def profile_callback(data, size, user_data):
    global profile_buffer
    if user_data == 1:
        ct.memmove(profile_buffer, data, size)
        event.set()

# Parametrize transmission
start_data = 0
data_width = 8
scanner_type = ct.c_int(0)

# Init profile buffer and timestamp info
timestamp = (ct.c_ubyte * 16)()
available_resolutions = (ct.c_uint * 4)()
available_interfaces = (ct.c_uint * 6)()
lost_profiles = ct.c_int()
shutter_opened = ct.c_double(0.0)
shutter_closed = ct.c_double(0.0)
profile_count = ct.c_uint(0)

# Callback function
get_profile_cb = llt.buffer_cb_func(profile_callback)
event = threading.Event()

# Null pointer if data not necessary
null_ptr_short = ct.POINTER(ct.c_ushort)()
null_ptr_int = ct.POINTER(ct.c_uint)()

# Create instance and set IP address
hLLT = llt.create_llt_device(llt.TInterfaceType.INTF_TYPE_ETHERNET)

# Get available interfaces
ret = llt.get_device_interfaces_fast(hLLT, available_interfaces, len(available_interfaces))
if ret < 1:
    raise ValueError("Error getting interfaces : " + str(ret))

ret = llt.set_device_interface(hLLT, available_interfaces[0], 0)
if ret < 1:
    raise ValueError("Error setting device interface: " + str(ret))

# Connect
ret = llt.connect(hLLT)
if ret < 1:
    raise ConnectionError("Error connect: " + str(ret))

# Get available resolutions
ret = llt.get_resolutions(hLLT, available_resolutions, len(available_resolutions))
if ret < 1:
    raise ValueError("Error getting resolutions : " + str(ret))

# Set max. resolution
resolution = available_resolutions[0]
ret = llt.set_resolution(hLLT, resolution)
if ret < 1:
    raise ValueError("Error getting resolutions : " + str(ret))

# Declare measuring data arrays
profile_buffer = (ct.c_ubyte*(resolution * data_width))()
x = np.empty(resolution, dtype=float)  # (ct.c_double * resolution)()
z = np.empty(resolution, dtype=float)  # (ct.c_double * resolution)()
x_p = x.ctypes.data_as(ct.POINTER(ct.c_double))
z_p = z.ctypes.data_as(ct.POINTER(ct.c_double))
intensities = (ct.c_ushort * resolution)()

# Partial profile struct
partial_profile_struct = llt.TPartialProfile(0, start_data, resolution, data_width)

# Scanner type
ret = llt.get_llt_type(hLLT, ct.byref(scanner_type))
if ret < 1:
    raise ValueError("Error scanner type: " + str(ret))

# Scanner type
ret = llt.set_resolution(hLLT, resolution)
if ret < 1:
    raise ValueError("Error setting resolution: " + str(ret))

# Set partial profile as profile config
ret = llt.set_profile_config(hLLT, llt.TProfileConfig.PARTIAL_PROFILE)
if ret < 1:
    raise ValueError("Error setting profile config: " + str(ret))

# Set trigger
ret = llt.set_feature(hLLT, llt.FEATURE_FUNCTION_TRIGGER, llt.TRIG_INTERNAL)
if ret < 1:
    raise ValueError("Error setting trigger: " + str(ret))

# Set exposure time
ret = llt.set_feature(hLLT, llt.FEATURE_FUNCTION_EXPOSURE_TIME, 1000)
if ret < 1:
    raise ValueError("Error setting exposure time: " + str(ret))

# Set idle time
ret = llt.set_feature(hLLT, llt.FEATURE_FUNCTION_IDLE_TIME, 3900)
if ret < 1:
    raise ValueError("Error idle time: " + str(ret))

# Set partial profile
ret = llt.set_partial_profile(hLLT, ct.byref(partial_profile_struct))
if ret < 1:
    raise ValueError("Error setting partial profile: " + str(ret))

# Register Callback
ret = llt.register_callback(hLLT, llt.TCallbackType.C_DECL, get_profile_cb, 1)
if ret < 1:
    raise ValueError("Error setting callback: " + str(ret))

# Start transfer
ret = llt.transfer_profiles(hLLT, llt.TTransferProfileType.NORMAL_TRANSFER, 1)
if ret < 1:
    raise ValueError("Error starting transfer profiles: " + str(ret))

# Warm-up time
time.sleep(0.1)

fig, ax = plt.subplots()
line, = ax.plot([], [], ".b", lw=2)
ax.grid()
ax.set_xlim(-60, 60)
ax.set_ylim(25, 350)
line.set_data(x, z)


def data_gen(*args):
    event.wait()
    fret = llt.convert_part_profile_2_values(hLLT, profile_buffer, ct.byref(partial_profile_struct), scanner_type, 0, 1,
                                             null_ptr_short, null_ptr_short, null_ptr_short, x_p, z_p, null_ptr_int, null_ptr_int)
    if fret & llt.CONVERT_X is 0 or fret & llt.CONVERT_Z is 0:
        raise ValueError("Error converting data: " + str(ret))

    for i in range(16):
        timestamp[i] = profile_buffer[resolution * data_width - 16 + i]

    llt.timestamp_2_time_and_count(timestamp, ct.byref(shutter_opened), ct.byref(shutter_closed), ct.byref(profile_count))
    event.clear()

    yield x, z


def update(data):
    ux, uz = data
    line.set_data(ux, uz)
    return line,

ani = animation.FuncAnimation(fig, update, frames=data_gen, interval=40)
plt.show()

ret = llt.transfer_profiles(hLLT, llt.TTransferProfileType.NORMAL_TRANSFER, 0)
if ret < 1:
    raise ValueError("Error stopping transfer profiles: " + str(ret))

# Disconnect
ret = llt.disconnect(hLLT)
if ret < 1:
    raise ConnectionAbortedError("Error while disconnect: " + str(ret))

# Delete
ret = llt.del_device(hLLT)
if ret < 1:
    raise ConnectionAbortedError("Error while delete: " + str(ret))


SAVE FILE

In [ ]:
import ctypes as ct
import time
import cv2
import numpy as np
import pyllt as llt
from datetime import datetime
from scipy.ndimage import median_filter
from config import config as CFG
from matplotlib import pyplot as plt
import threading

class Laser:
    def __init__(self) -> None:
        self.hllt = llt.create_llt_device(llt.TInterfaceType.INTF_TYPE_ETHERNET)
        self.IDLE_TIME = CFG.IDLE_TIME
        self.scanner_type = ct.c_int(0)
        self.exposure_time = CFG.EXPOSURE_TIME
        self.null_ptr_short = ct.POINTER(ct.c_ushort)()
        self.null_ptr_int = ct.POINTER(ct.c_uint)()
        self.available_resolutions = (ct.c_uint * 4)()
        self.available_interfaces = (ct.c_uint * 6)()
        self.lost_profiles = ct.c_int()
        self.profile_buffer = None
        self.resolution = None
        self.data_width = 8
        self.start_data = 0

    def profile_callback(self, data, size, user_data):
        if user_data == 1:
            ct.memmove(self.profile_buffer, data, size)
            self.event.set()

    def connect(self):
        ret = llt.get_device_interfaces_fast(self.hllt, self.available_interfaces, len(self.available_interfaces))
        if ret < 1:
            raise ValueError(f"Error getting interfaces: {ret}")

        ret = llt.set_device_interface(self.hllt, self.available_interfaces[0], 0)
        if ret < 1:
            raise ValueError(f"Error setting device interface: {ret}")

        ret = llt.connect(self.hllt)
        if ret < 1:
            raise ConnectionError(f"Error connect: {ret}")

        ret = llt.get_resolutions(self.hllt, self.available_resolutions, len(self.available_resolutions))
        if ret < 1:
            raise ValueError(f"Error getting resolutions: {ret}")

        self.resolution = self.available_resolutions[0]
        ret = llt.set_resolution(self.hllt, self.resolution)
        if ret < 1:
            raise ValueError(f"Error setting resolution: {ret}")

        self.profile_buffer = (ct.c_ubyte * (self.resolution * self.data_width))()
        self.X_value = (ct.c_double * self.resolution)()
        self.Z_value = (ct.c_double * self.resolution)()
        self.intensities = (ct.c_ushort * self.resolution)()

        ret = llt.get_llt_type(self.hllt, ct.byref(self.scanner_type))
        if ret < 1:
            raise ValueError(f"Error scanner type: {ret}")

        ret = llt.set_profile_config(self.hllt, llt.TProfileConfig.PARTIAL_PROFILE)
        if ret < 1:
            raise ValueError(f"Error setting profile config: {ret}")

        ret = llt.set_feature(self.hllt, llt.FEATURE_FUNCTION_TRIGGER, llt.TRIG_INTERNAL)
        if ret < 1:
            raise ValueError(f"Error setting trigger: {ret}")

        ret = llt.set_feature(self.hllt, llt.FEATURE_FUNCTION_EXPOSURE_TIME, self.exposure_time)
        if ret < 1:
            raise ValueError(f"Error setting exposure time: {ret}")

        ret = llt.set_feature(self.hllt, llt.FEATURE_FUNCTION_IDLE_TIME, self.IDLE_TIME)
        if ret < 1:
            raise ValueError(f"Error setting idle time: {ret}")

        partial_profile_struct = llt.TPartialProfile(0, self.start_data, self.resolution, self.data_width)
        ret = llt.set_partial_profile(self.hllt, ct.byref(partial_profile_struct))
        if ret < 1:
            raise ValueError(f"Error setting partial profile: {ret}")

        self.get_profile_cb = llt.buffer_cb_func(self.profile_callback)
        self.event = threading.Event()

        ret = llt.register_callback(self.hllt, llt.TCallbackType.C_DECL, self.get_profile_cb, 1)
        if ret < 1:
            raise ValueError(f"Error setting callback: {ret}")

    def start_scan(self):
        print("Start transfer!")
        ret = llt.transfer_profiles(self.hllt, llt.TTransferProfileType.NORMAL_TRANSFER, 1)
        if ret < 1:
            raise ValueError(f"Error starting transfer profiles: {ret}")
        print("Transfering...")

        # Open file to save profiles
        file = open("profile_data.txt", "w")

        def data_gen():
            while True:
                self.event.wait()
                fret = llt.convert_part_profile_2_values(
                    self.hllt, self.profile_buffer, ct.byref(partial_profile_struct), self.scanner_type, 0, 1,
                    self.null_ptr_short, self.null_ptr_short, self.null_ptr_short, self.X_value, self.Z_value, self.null_ptr_int, self.null_ptr_int
                )
                if fret & llt.CONVERT_X == 0 or fret & llt.CONVERT_Z == 0:
                    raise ValueError("Error converting data: " + str(fret))

                # Save profile data to file
                for i in range(self.resolution):
                    file.write(f"{self.X_value[i]}\t{self.Z_value[i]}\n")

                file.write("\n")  # Separate profiles by an empty line

                self.event.clear()

        data_gen_thread = threading.Thread(target=data_gen)
        data_gen_thread.start()

        time.sleep(CFG.SLEEP)
        print("Finish transfer!")

        ret = llt.transfer_profiles(self.hllt, llt.TTransferProfileType.NORMAL_TRANSFER, 0)
        if ret < 1:
            raise ValueError(f"Error stopping transfer profiles: {ret}")

        data_gen_thread.join()
        file.close()
        print("Finish scanning!")

    def disconnect(self):
        ret = llt.disconnect(self.hllt)
        if ret < 1:
            raise ConnectionAbortedError(f"Error while disconnecting: {ret}")

        ret = llt.del_device(self.hllt)
        if ret < 1:
            raise ConnectionAbortedError(f"Error while deleting: {ret}")

if __name__ == '__main__':
    laser = Laser()
    laser.connect()
    laser.start_scan()
    laser.disconnect()


COMBINE

In [ ]:
import ctypes as ct
import time
import cv2
import numpy as np
import pyllt as llt
from datetime import datetime
from scipy.ndimage import median_filter
from config import config as CFG
from matplotlib import pyplot as plt
import threading

class Laser:
    def __init__(self) -> None:
        self.hllt = llt.create_llt_device(llt.TInterfaceType.INTF_TYPE_ETHERNET)
        self.IDLE_TIME = CFG.IDLE_TIME
        self.scanner_type = ct.c_int(0)
        self.exposure_time = CFG.EXPOSURE_TIME
        self.null_ptr_short = ct.POINTER(ct.c_ushort)()
        self.null_ptr_int = ct.POINTER(ct.c_uint)()
        self.available_resolutions = (ct.c_uint * 4)()
        self.available_interfaces = (ct.c_uint * 6)()
        self.lost_profiles = ct.c_int()
        self.profile_buffer = None
        self.resolution = None
        self.data_width = 8
        self.start_data = 0
        self.number_of_profiles = CFG.NUMBER_OF_PROFILES

    def profile_callback(self, data, size, user_data):
        if user_data == 1:
            ct.memmove(self.profile_buffer, data, size)
            self.event.set()

    def connect(self):
        ret = llt.get_device_interfaces_fast(self.hllt, self.available_interfaces, len(self.available_interfaces))
        if ret < 1:
            raise ValueError(f"Error getting interfaces: {ret}")

        ret = llt.set_device_interface(self.hllt, self.available_interfaces[0], 0)
        if ret < 1:
            raise ValueError(f"Error setting device interface: {ret}")

        ret = llt.connect(self.hllt)
        if ret < 1:
            raise ConnectionError(f"Error connect: {ret}")

        ret = llt.get_resolutions(self.hllt, self.available_resolutions, len(self.available_resolutions))
        if ret < 1:
            raise ValueError(f"Error getting resolutions: {ret}")

        self.resolution = self.available_resolutions[0]
        ret = llt.set_resolution(self.hllt, self.resolution)
        if ret < 1:
            raise ValueError(f"Error setting resolution: {ret}")

        self.profile_buffer = (ct.c_ubyte * (self.resolution * self.data_width))()
        self.X_value = (ct.c_double * self.resolution)()
        self.Z_value = (ct.c_double * self.resolution)()
        self.intensities = (ct.c_ushort * self.resolution)()

        ret = llt.get_llt_type(self.hllt, ct.byref(self.scanner_type))
        if ret < 1:
            raise ValueError(f"Error scanner type: {ret}")

        ret = llt.set_profile_config(self.hllt, llt.TProfileConfig.PARTIAL_PROFILE)
        if ret < 1:
            raise ValueError(f"Error setting profile config: {ret}")

        ret = llt.set_feature(self.hllt, llt.FEATURE_FUNCTION_TRIGGER, llt.TRIG_INTERNAL)
        if ret < 1:
            raise ValueError(f"Error setting trigger: {ret}")

        ret = llt.set_feature(self.hllt, llt.FEATURE_FUNCTION_EXPOSURE_TIME, self.exposure_time)
        if ret < 1:
            raise ValueError(f"Error setting exposure time: {ret}")

        ret = llt.set_feature(self.hllt, llt.FEATURE_FUNCTION_IDLE_TIME, self.IDLE_TIME)
        if ret < 1:
            raise ValueError(f"Error setting idle time: {ret}")

        partial_profile_struct = llt.TPartialProfile(0, self.start_data, self.resolution, self.data_width)
        ret = llt.set_partial_profile(self.hllt, ct.byref(partial_profile_struct))
        if ret < 1:
            raise ValueError(f"Error setting partial profile: {ret}")

        self.get_profile_cb = llt.buffer_cb_func(self.profile_callback)
        self.event = threading.Event()

        ret = llt.register_callback(self.hllt, llt.TCallbackType.C_DECL, self.get_profile_cb, 1)
        if ret < 1:
            raise ValueError(f"Error setting callback: {ret}")

    def start_scan(self):
        print("Start transfer!")
        ret = llt.transfer_profiles(self.hllt, llt.TTransferProfileType.NORMAL_TRANSFER, 1)
        if ret < 1:
            raise ValueError(f"Error starting transfer profiles: {ret}")
        print("Transfering...")

        # Open file to save profiles
        file = open("profile_data.txt", "w")

        def data_gen():
            profile_count = 0
            while profile_count < self.number_of_profiles:
                self.event.wait()
                fret = llt.convert_part_profile_2_values(
                    self.hllt, self.profile_buffer, ct.byref(partial_profile_struct), self.scanner_type, 0, 1,
                    self.null_ptr_short, self.null_ptr_short, self.null_ptr_short, self.X_value, self.Z_value, self.null_ptr_int, self.null_ptr_int
                )
                if fret & llt.CONVERT_X == 0 or fret & llt.CONVERT_Z == 0:
                    raise ValueError("Error converting data: " + str(fret))

                # Save profile data to file
                for i in range(self.resolution):
                    file.write(f"{self.X_value[i]}\t{self.Z_value[i]}\n")

                file.write("\n")  # Separate profiles by an empty line
                profile_count += 1
                self.event.clear()

        data_gen_thread = threading.Thread(target=data_gen)
        data_gen_thread.start()

        data_gen_thread.join()
        file.close()
        print("Finish scanning!")

        ret = llt.transfer_profiles(self.hllt, llt.TTransferProfileType.NORMAL_TRANSFER, 0)
        if ret < 1:
            raise ValueError(f"Error stopping transfer profiles: {ret}")

    def disconnect(self):
        ret = llt.disconnect(self.hllt)
        if ret < 1:
            raise ConnectionAbortedError(f"Error while disconnecting: {ret}")

        ret = llt.del_device(self.hllt)
        if ret < 1:
            raise ConnectionAbortedError(f"Error while deleting: {ret}")

if __name__ == '__main__':
    laser = Laser()
    laser.connect()
    laser.start_scan()
    laser.disconnect()
